# 58. 원판 중심이 가까운 무릎 높이를 지나는 시점

트레이너가 판독한 검증 6명의 측면 영상에서 두 점의 상대 높이를 그린다. 첫 실행은 5편에서 교차를 찾았고 수동 라벨과 평균 절대 차이는 1.6프레임(55 ms)이었다. a03은 판정을 보류했다.

이번에는 이륙과 바 접촉을 수동으로 지정하고 **그 사이에서 무릎 통과 한 시점만** 찾는다. 원판은 기존 베이스라인, 자세는 원본 YOLO26n-pose, 두 입력 크기는 640이다. 무릎 통과 정답은 후보 선택 이후에만 사용한다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").is_file())
sys.path.insert(0, str(root))
from wlcv.evaluation.knee_pass_height import run, plot_curves, plot_case
from wlcv.annotation.plate_blur import REVIEW_FILE

# 저장된 수치로 다시 볼 때는 False. True면 Mac에서 모델을 다시 실행해 새 폴더를 만든다.
rerun = False
folder = root / "outputs/knee_pass_height/20261126-000518-592842"
if rerun or not folder.is_dir():
    folder = run(device="mps")
frames = pd.read_csv(folder / "frames.csv")
summary = pd.read_csv(folder / "summary.csv")
print(folder)

## 1. 상대 높이와 무릎 통과

화면의 y는 아래로 증가한다. **무릎 y − 원판 박스 중심 y**가 음수면 원판 중심이 무릎 아래에 있다. 이 값이 연속된 두 프레임에서 음수 → 0 이상으로 바뀌면 뒤 프레임을 선택한다. 첫 관측부터 0 이상이거나 교차 사이가 비어 있으면 보류한다. 결측은 보간하지 않는다.

원판은 신뢰도 0.10 이상 직접 검출과 기존 0.5배 시간 연결을 쓴다. 자세는 수행자가 하나면 선택하고 여러 명이면 직전 수행자와 IoU 0.50 이상인 후보가 유일할 때 선택한다. 가까운 무릎 신뢰도가 0.50 미만이면 해당 점은 곡선에서 비운다.

In [ ]:
curves = plot_curves(frames, summary)
curves.savefig(folder / "height_curves.png", dpi=140)
plt.show()

## 2. 판정 가능 범위와 오차

오차는 규칙 프레임에서 수동 프레임을 뺀 값이다. 음수는 일찍, 양수는 늦게 찾았다는 뜻이다. 시간 차이는 각 영상 프레임의 실제 타임스탬프로 구한다. 보류 영상은 전체 영상 수에 포함하고 오차 평균에서는 제외한다.

In [ ]:
result = summary[["video", "manual", "automatic", "error_frames", "error_ms",
                  "valid_frames", "window_frames", "status"]].copy()
result.columns = ["영상", "수동", "규칙", "차이(프레임)", "차이(ms)", "동시 관측", "구간 장수", "판정"]
display(result.round(1))
scored = summary.dropna(subset=["automatic"])
print(f"판정 {len(scored)}/{len(summary)}편")
print(f"평균 절대 차이 {scored.error_frames.abs().mean():.2f}프레임, {scored.error_ms.abs().mean():.1f} ms")

## 3. 수동 시점과 규칙 시점의 원본 비교

영상 ID를 바꿔 각 선수의 원본을 확인한다. 노란 십자는 예측 원판 중심, 청록색 ×와 점선은 예측한 가까운 무릎과 그 높이다. 원판 뒤에 가려진 무릎 위치는 육안도 확신하기 어렵다.

In [ ]:
video = "c04_sn_f_01"
image = plot_case(frames, summary, video)
image.savefig(folder / f"{video[:3]}_frames.jpg", dpi=125)
plt.show()
case = summary.set_index("video").loc[video]
window = frames.loc[frames.video.eq(video) & frames.frame.between(case.lift, case.contact)]
display(window[["frame", "plate_status", "plate_y", "person", "knee_score", "knee_status", "knee_y", "gap_px"]].round(2))

## 4. 이미 직접 찍은 허브로 점검

수동 통과 프레임에서만 원판 중심을 기존 수동 허브로 바꿔 같은 관절과의 높이 차이를 다시 본다. 이 표는 원인 점검용이다. 무릎은 여전히 모델 예측이므로 해부학적 위치 오차는 여기서 확정할 수 없다.

In [ ]:
at_manual = summary[["video", "manual"]].merge(
    frames, left_on=["video", "manual"], right_on=["video", "frame"], validate="one_to_one")
hubs = pd.read_csv(REVIEW_FILE)
hubs = hubs[hubs.hub_source.isin(["clicked", "previous_blind"])]
audit = at_manual.merge(hubs[["video_id", "frame_index", "manual_y"]], how="left",
                        left_on=["video", "frame"], right_on=["video_id", "frame_index"],
                        validate="one_to_one")
audit["plate_minus_manual_hub_px"] = audit.plate_y - audit.manual_y
audit["knee_minus_manual_hub_px"] = audit.knee_y - audit.manual_y
audit = audit[["video", "frame", "gap_px", "manual_y", "plate_minus_manual_hub_px", "knee_minus_manual_hub_px"]]
audit.to_csv(folder / "manual_hub_check.csv", index=False)
display(audit.round(2))

## 5. 시점을 바꾸면 구간 상승량이 얼마나 달라질까?

이륙 프레임과 모델을 고정하고 끝점만 바꾼다. 두 열 모두 모델 박스 중심에서 계산한 화면 픽셀이다. 수동 시점 열도 수동 허브로 잰 실제 상승량을 뜻하지 않는다.

In [ ]:
rise = summary[["video", "manual_event_rise_px", "auto_event_rise_px"]].copy()
rise["difference_px"] = rise.auto_event_rise_px - rise.manual_event_rise_px
display(rise.round(2))

첫 실행의 c04는 27번에서 31번으로 끝점이 이동해 상승량이 약 71 px 커졌다. c04·c08은 수동 허브를 넣어도 수동 시점에서 예측 무릎보다 허브가 낮았다. 현재 규칙은 사람이 다시 볼 후보를 찾는 데 사용한다. 다음 검수에서는 수동 시점의 기준과 예측 무릎의 판독 가능 여부를 먼저 확인한다.